In [1]:
from langchain_openai.chat_models import ChatOpenAI, AzureChatOpenAI
from pydantic import BaseModel
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate 
import os

In [2]:
load_dotenv()

True

In [3]:
text = """Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structure.
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future."""

In [4]:
# pydantic class for structured output

class Chunk(BaseModel): 
    
    chunk_text: str
    summary: str
    
    
class Chunker(BaseModel):
    
    chunks: list[Chunk]

In [5]:
# # define model

# model = AzureChatOpenAI(model="gpt-4o-mini")

# llm_chunker = model.with_structured_output(schema=Chunker)



model = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment="gpt-4o-mini",
)

llm_chunker = model.with_structured_output(schema=Chunker)

In [6]:
# prompt for chunking

prompt = ChatPromptTemplate(messages=[
    ("system", 
     """You are an expert Text Chunker that splits the given text and outputs them as a 
     list of strings. You understand the natural topic boundaries of text and 
     also do not change the existing text. You just split the text where ever applicable.
     Once you create the chunk, you also generate a 1-2 line summary of the chunk also"""),
    ("human",
     "Split the given text into chunks\nText: {text}")
], input_variables=["text"])

In [7]:
# chunking through llm

model_chain = prompt | llm_chunker

response = model_chain.invoke({"text": text})

In [8]:
response

Chunker(chunks=[Chunk(chunk_text='Artificial intelligence is transforming technology and shaping the future.', summary='This chunk highlights the impact of artificial intelligence on technology and future developments.'), Chunk(chunk_text='Machine learning algorithms are becoming more sophisticated every day.', summary='This chunk discusses the increasing sophistication of machine learning algorithms.'), Chunk(chunk_text='Deep learning models can now process vast amounts of data efficiently.', summary='This chunk addresses the efficiency of deep learning models in handling large data.'), Chunk(chunk_text="Neural networks are inspired by the human brain's structure.", summary="This chunk explains that neural networks draw inspiration from the brain's architecture."), Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques.', summary='This chunk emphasizes the importance of using fresh ingredients and techniques in pasta recipes.'), Chunk(chunk_te

In [9]:
response.chunks

[Chunk(chunk_text='Artificial intelligence is transforming technology and shaping the future.', summary='This chunk highlights the impact of artificial intelligence on technology and future developments.'),
 Chunk(chunk_text='Machine learning algorithms are becoming more sophisticated every day.', summary='This chunk discusses the increasing sophistication of machine learning algorithms.'),
 Chunk(chunk_text='Deep learning models can now process vast amounts of data efficiently.', summary='This chunk addresses the efficiency of deep learning models in handling large data.'),
 Chunk(chunk_text="Neural networks are inspired by the human brain's structure.", summary="This chunk explains that neural networks draw inspiration from the brain's architecture."),
 Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques.', summary='This chunk emphasizes the importance of using fresh ingredients and techniques in pasta recipes.'),
 Chunk(chunk_text='Italia

In [10]:
len(response.chunks)

12

In [11]:
chunks = response.chunks

In [12]:
from termcolor import COLORS, colored
from random import choice

In [13]:
def display_chunks(chunks):
    colors_list = list(COLORS.keys())[2:8]
    print(f"Total Number of Chunks: {len(chunks)}")
    
    for num, chunk in enumerate(chunks, 1):
        print(f"Chunk {num}: Length {len(chunk)} chars")
        print(colored(text=chunk, color=choice(colors_list)), end="\n\n")

In [14]:
display_chunks([chunk.chunk_text for chunk in chunks])

Total Number of Chunks: 12
Chunk 1: Length 74 chars
Artificial intelligence is transforming technology and shaping the future.

Chunk 2: Length 70 chars
Machine learning algorithms are becoming more sophisticated every day.

Chunk 3: Length 70 chars
Deep learning models can now process vast amounts of data efficiently.

Chunk 4: Length 60 chars
Neural networks are inspired by the human brain's structure.

Chunk 5: Length 79 chars
The best pasta recipes include fresh ingredients and proper cooking techniques.

Chunk 6: Length 66 chars
Italian cuisine emphasizes quality olive oil and regional cheeses.

Chunk 7: Length 76 chars
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.

Chunk 8: Length 59 chars
Cooking pasta al dente ensures the best texture and flavor.

Chunk 9: Length 49 chars
Climate change is affecting ecosystems worldwide.

Chunk 10: Length 72 chars
Rising temperatures are causing glaciers to melt at unprecedented rates.

Chunk 11: Length 75 chars
S

In [15]:
response.chunks[2]

Chunk(chunk_text='Deep learning models can now process vast amounts of data efficiently.', summary='This chunk addresses the efficiency of deep learning models in handling large data.')

In [16]:
# create documents from chunks

docs = [Document(page_content=chunk.chunk_text, metadata={"summary": chunk.summary}) for chunk in chunks]

In [17]:
print(docs)

[Document(metadata={'summary': 'This chunk highlights the impact of artificial intelligence on technology and future developments.'}, page_content='Artificial intelligence is transforming technology and shaping the future.'), Document(metadata={'summary': 'This chunk discusses the increasing sophistication of machine learning algorithms.'}, page_content='Machine learning algorithms are becoming more sophisticated every day.'), Document(metadata={'summary': 'This chunk addresses the efficiency of deep learning models in handling large data.'}, page_content='Deep learning models can now process vast amounts of data efficiently.'), Document(metadata={'summary': "This chunk explains that neural networks draw inspiration from the brain's architecture."}, page_content="Neural networks are inspired by the human brain's structure."), Document(metadata={'summary': 'This chunk emphasizes the importance of using fresh ingredients and techniques in pasta recipes.'}, page_content='The best pasta re

In [18]:
print(docs[1])

page_content='Machine learning algorithms are becoming more sophisticated every day.' metadata={'summary': 'This chunk discusses the increasing sophistication of machine learning algorithms.'}
